# 29 · Ngân sách token của Qwen2.5-VL-3B trên ViVQA + ViTextVQA (W15, góp ý Review 2 — G1)

**Vấn đề (Review 2, 29/10/2026):** hội đồng hỏi mỗi câu hỏi tốn **bao nhiêu token** (ảnh, câu hỏi, dòng OCR) và tổng đó so với **sức chứa** (context window) của model ra sao. Slide kiến trúc mới ghi "256–512 token ảnh", chưa có số đo trên dữ liệu thật.

**Notebook này đo chính xác** (không ước lượng theo số ký tự), trên 4 tập: ViVQA test / train, ViTextVQA test (10.028 câu) / train (toàn bộ), mỗi tập 2 kiểu prompt (không OCR, có dòng OCR):
- **Token ảnh** = đi đúng đường suy luận: `qwen_vl_utils` → image processor của Qwen (min 256, max 512 token). Kèm số token **nếu không chặn** (độ phân giải gốc) và **nếu chặn ở 1.280**.
- **Token chữ** = tokenizer Qwen trên đúng chuỗi chat template (system + câu hỏi [+ dòng OCR]).
- **Token đáp án** (tập train) = như lúc train QLoRA.
- **Kiểm chứng:** 150 câu ngẫu nhiên mỗi tập được chạy lại bằng **nguyên đường processor như lúc suy luận**; số token phải trùng hệt từng câu.
- So với **context window** đọc từ `config.json` của model, và **KV cache** (bộ nhớ cho mỗi token).

**Runtime:** **CPU** (không cần GPU, gần như không tốn CU). Khoảng 30–45 phút (phần lâu nhất là chép/giải nén ảnh).

**Anh chỉ cần:** Runtime → Change runtime type → **CPU** → Run all, rồi gửi Claude **output mục 5**. Máy tự trả khi xong.

## 1. Thiết lập môi trường

Mount Drive, lấy code mới nhất từ GitHub, kiểm có script `token_budget.py`.

*Kỳ vọng: commit mới nhất (W15 token budget), `Runtime: CPU`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, subprocess
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('src/eval/token_budget.py'), '❌ Chưa có src/eval/token_budget.py -> commit W15 chưa lên GitHub, báo Claude'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
OUT = f'{DRIVE_ROOT}/experiments/W15_token_budget'

def run_cmd(cmd, check=True):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {rc}) -> xem log ở trên, gửi Claude')
    return rc

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU, '(CPU là đủ cho notebook này)')
print('✅ Thiết lập xong')

## 2. Cài thư viện (≈ 2 phút)

Chỉ cần tokenizer + image processor của Qwen, **không** nạp model, nên không cần bitsandbytes / peft / GPU. Dùng đúng phiên bản đã chạy mọi thí nghiệm (`requirements-colab.txt`): `transformers==5.17.0`, `qwen-vl-utils==0.0.14`. Thêm `pyvi` (tách từ tiếng Việt) để chạy selftest của script chuẩn hoá/tách từ (G2–G3).

*Kỳ vọng: `transformers 5.17.0 | qwen_vl_utils OK | torch …`.*

In [ ]:
run_cmd('pip install -q transformers==5.17.0 qwen-vl-utils==0.0.14 pyyaml pyvi')
import importlib, transformers, torch
import qwen_vl_utils  # noqa: F401
assert transformers.__version__ == '5.17.0', transformers.__version__
print('transformers', transformers.__version__, '| qwen_vl_utils OK | torch', torch.__version__)

## 3. Kiểm tra trước khi chạy (≈ 1 phút)

- **(a)** Selftest `token_budget` (logic đếm, công thức token ảnh, KV cache) và `metric_tokenization`.
- **(b)** Đủ 4 file dữ liệu và 2 OCR cache trên Drive (cache `paddledet_vietocr.json` — đúng cache của các run chính thức).

*Kỳ vọng: `token_budget SELFTEST OK`, `metric_tokenization SELFTEST OK`, 6 dòng `✅`.*

In [ ]:
for mod, need in {'src.eval.token_budget': 'token_budget SELFTEST OK',
                  'src.eval.metric_tokenization': 'metric_tokenization SELFTEST OK'}.items():
    o = subprocess.run(f'python -W ignore -m {mod} --selftest', shell=True, capture_output=True, text=True)
    assert need in o.stdout, (mod, o.stdout[-800:], o.stderr[-800:])
    print(need)

from src.eval.token_budget import SPLITS, OCR_BACKEND
from src.data.ocr import default_cache_path
for key, spec in SPLITS.items():
    p = f"{DRIVE_ROOT}/{spec['data_rel']}"
    assert os.path.exists(p), f'❌ thiếu {p}'
    print(f'✅ {key}: {spec["data_rel"]}')
for ds in ('vivqa', 'vitextvqa_official'):
    p = default_cache_path(DRIVE_ROOT, ds, OCR_BACKEND)
    assert os.path.exists(p), f'❌ thiếu OCR cache {p}'
    print(f'✅ OCR cache {ds}: {os.path.basename(p)}')

## 4. Đo ngân sách token (≈ 25–40 phút)

`python -m src.eval.token_budget --run`:
1. Tải tokenizer + processor của Qwen2.5-VL-3B (đúng revision đã dùng ở W07), đọc context window + số lớp/KV-head từ `config.json`.
2. Chép ảnh ViVQA về `/content` và giải nén ảnh ViTextVQA (như các notebook eval).
3. Mỗi tập × mỗi kiểu prompt: đo từng câu → kiểm chứng 150 câu bằng đường processor đầy đủ → ghi CSV từng câu + thống kê.

Nếu đường nhanh lệch ở tập nào, script **tự đo lại** tập đó bằng processor (chậm hơn) và ghi rõ trong `validation.json` — không cần làm gì.

*Kỳ vọng: 8 dòng dạng `vitextvqa_test|ocr: input p50/p95/max = …/…/… | ảnh p50 … | chặn 512: …% | kiểm chứng lệch 0/150`, cuối cùng `✅ xong …`.*

In [ ]:
run_cmd(f'python -u -W ignore -m src.eval.token_budget --run --drive_root "{DRIVE_ROOT}" --out "{OUT}"')

## 5. Kết quả — **gửi Claude toàn bộ output cell này**

In bảng tóm tắt (`summary.md`) và khối số liệu gọn để Claude kiểm và đưa lên slide kiến trúc (G1).

*Kỳ vọng: bảng 8 dòng + dòng `Kiểm chứng … lệch 0 câu (ĐẠT)` + khối `=== GỬI CLAUDE ===`.*

In [ ]:
print(open(f'{OUT}/summary.md', encoding='utf-8').read())
S = json.load(open(f'{OUT}/summary.json', encoding='utf-8'))
keep = ['n_questions', 'n_images', 'image_tokens', 'pct_images_capped_512', 'pct_images_raised_to_256',
        'native_image_tokens', 'image_tokens_if_1280_cap', 'question_tokens', 'ocr_line_tokens_when_present',
        'pct_questions_with_ocr_line', 'text_tokens', 'input_tokens', 'answer_tokens', 'sequence_train',
        'longest_sequence', 'pct_of_context_longest', 'kv_cache_mb_longest', 'image_share_of_input_p50']
print('=== GỬI CLAUDE ===')
print(json.dumps({'model': S['model'], 'versions': S['versions'], 'elapsed_sec': S['elapsed_sec'],
                  'validation': [{k: v[k] for k in ('split', 'mode', 'n_checked', 'n_mismatch', 'method')} for v in S['validation']['per_split']],
                  'results': {k: {kk: v[kk] for kk in keep if kk in v} for k, v in S['results'].items()}},
                 ensure_ascii=False))
print('=== HẾT ===')
print('Files:', sorted(os.listdir(OUT)))

## 6. Tự trả máy

Kết quả đã nằm trên Drive (`experiments/W15_token_budget/`). Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: gửi lỗi cho Claude rồi bấm **Runtime → Disconnect and delete runtime**.

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()